# Scoring experiments
> Compare sampling strategies with the definitions of `07_scoring.ipynb`

Each comparison runs many campaigns and takes up to a minute. The leading underscore keeps this notebook out of `nbdev-test` and the docs: run it by hand, and record results in `scoring-plan.md`. The first cell runs the code cells of `07_scoring.ipynb`, lessons included, to load its definitions.

In [ ]:
from fastcore.nbio import read_nb
for c in read_nb('07_scoring.ipynb').cells:
    if c.cell_type == 'code': get_ipython().run_cell(c.source, silent=True).raise_error()

## Baseline comparison

In [ ]:
strategies = {'equal': equal(), 'prior only': mcdm(fname_raster, [PriorCB(fname_raster)], [True], [1]),
              'max only': mcdm(fname_raster, [MaxCB()], [True], [1]), 'std only': mcdm(fname_raster, [StdCB()], [True], [1]),
              'how-to': mcdm(fname_raster), 'neyman': neyman(), 'oracle': oracle(fname_raster)}
cmp = compare(campaign, strategies, budgets=[150] * 6, seeds=range(5))


Mean Wasserstein distance over units, averaged over seeds, by round:

In [ ]:
cmp.groupby(['strategy', 'round']).mean_wasserstein.mean().unstack().round(4)

## Minimum number of samples

Five strategies under four minimums, three seeds each: no minimum, one sample per unit in every round, and one or two samples per unit over the whole campaign:

In [ ]:
factories = {'equal': equal, 'how-to': lambda m: mcdm(fname_raster, min=m),
             'std only': lambda m: mcdm(fname_raster, [StdCB()], [True], [1], min=m),
             'neyman': neyman, 'oracle': lambda m: oracle(fname_raster, min=m)}
minimums = {'none': lambda f: f(0), '1 per round': lambda f: f(1),
            '1 in total': lambda f: cumulative_min(f(0), 1), '2 in total': lambda f: cumulative_min(f(0), 2)}
cmp_min = pd.concat({name: compare(campaign, {k: make(f) for k, f in factories.items()}, budgets=[150] * 6, seeds=range(3))
                     for name, make in minimums.items()}, names=['minimum'])

Mean Wasserstein distance after round 6, averaged over seeds:

In [ ]:
cmp_min.xs(6, level='round').groupby(['strategy', 'minimum']).mean_wasserstein.mean().unstack()[list(minimums)].round(4)

Without a minimum per round, the oracle improves, and every strategy that estimates from the samples gets worse. `std only` is the extreme case: `StdCB` gives a unit with one sample a standard deviation of 0, and the strategy never chooses that unit again. With a minimum of 1 in total and seed 0, 25 units still have a single sample after six rounds.

## Decision score

With a permissible level of 0.8 and 10% accepted, 35 of the 100 units should be restricted. The raster is smooth: only 9 units have between 2% and 30% of their cells above the level, so few decisions are close. The strategies of the baseline comparison, one sample per unit per round, 5 seeds:

In [ ]:
dcampaign = Campaign(fname_raster, gdf_grid, score=decisions)
test_eq((dcampaign.score(campaign.samples.iloc[:0], dcampaign.truth).true_action == 'restrict').sum(), 35)
cmp_dec = compare(dcampaign, strategies, budgets=[150] * 6, seeds=range(5), summary=summarise_decisions)

Wrong decisions (needless plus missed), averaged over seeds, by round:

In [ ]:
cmp_dec.groupby(['strategy', 'round']).wrong.mean().unstack()

After six rounds, every strategy gets between 2.6 and 4.2 of the 100 decisions wrong, mostly missed restrictions, and the standard deviation over seeds reaches 2.7. Most differences are within that noise. The how-to strategy no longer trails equal allocation, and is ahead of it in rounds 2 and 3. With only 9 close units, this raster cannot separate the strategies. The `oracle` here targets the true standard deviation, not the decisions.

## Decision-aware strategies

Five strategies on two ground truths, 20 seeds each, 6 rounds of 150 samples. Every strategy gets a minimum of one sample per unit over the whole campaign, like the dashboard's minimum number of samples set to 1, and targets the rest freely. `ground-truth-01` at a permissible level of 0.8 has 35 units to restrict and 9 near the boundary (between 2% and 30% of cells above the level). `ground-truth-02` at 0.473 has 48 units to restrict and 32 near the boundary. The accepted proportion is 10%:

In [ ]:
from functools import partial

In [ ]:
def decision_strategies(fname, level, accepted=0.1):
    "The five strategies, each with a minimum of one sample per unit over the campaign."
    return {name: cumulative_min(s, 1) for name, s in {
        'equal': equal(0), 'how-to': mcdm(fname, min=0), 'neyman': neyman(0),
        'bayes': bayes(level, accepted, 0), 'oracle': decision_oracle(fname, level, accepted, 0)}.items()}

f02 = 'files/ground-truth-02-4326-simulated.tif'
truths = {'gt-01 at 0.8': (fname_raster, gdf_grid, 0.8), 'gt-02 at 0.473': (f02, gridder(f02, nrows=10, ncols=10), 0.473)}
cmp_bayes = pd.concat({name: compare(Campaign(f, g, score=partial(decisions, level=l)), decision_strategies(f, l),
                                     budgets=[150] * 6, seeds=range(20), summary=summarise_decisions)
                       for name, (f, g, l) in truths.items()}, names=['truth'])

Wrong decisions, averaged over seeds, by round:

In [ ]:
cmp_bayes.groupby(['truth', 'strategy', 'round']).wrong.mean().unstack().round(1)

With decisions as the score, the strategies separate. After round 6, `bayes` beats equal allocation on both ground truths (2.5 against 3.2 wrong decisions on `gt-01`, 7.8 against 10.0 on `gt-02`), the how-to strategy lies between them, and Neyman allocation is no better than equal. The decision oracle is best (1.4 and 6.5), which leaves room for a better information layer. The standard deviation over seeds is up to 3.2, so the standard error of each mean is below 0.8. The oracle on `gt-02` gets worse from round 2 (6.6) to round 3 (9.0). The dashboard's rule causes it: with 10% accepted, the 10th sample raises the number of exceedances needed to restrict from 1 to 2. For a unit whose true share is 12%, the probability of a wrong decision is 32% with 9 samples and 66% with 10:

In [ ]:
{n: round(float(binom.cdf(np.floor(0.1 * n), n, 0.12)), 3) for n in range(5, 13)}

## Fukushima windows

For internal checks only: these windows come from the 2011 Fukushima air dose raster in `_data`, which git ignores, and nothing derived from them is to be published. The values are log10 of the air dose rate in µSv/h. Each window is a 10 km square, cut into a 10 x 10 grid of 1 km units: one near the plant on the land side, one in the north-west plume around Iitate, and one far to the south-west. None has missing cells. `cut_window` writes them to `_data/windows` as dose rates in µSv/h. `LINEAR1`, which the how-to strategy uses, rejects negative values, and the log10 of any dose rate below 1 µSv/h is negative:

In [ ]:
import os, rasterio
from rasterio.windows import from_bounds
from pyproj import Transformer

In [ ]:
def cut_window(src_path, dst_path, cx, cy, half=5000):
    "Write the square of `src_path` centred on (`cx`, `cy`), `2 * half` metres wide, to `dst_path`, as 10 to the power of its values."
    with rasterio.open(src_path) as src:
        w = from_bounds(cx - half, cy - half, cx + half, cy + half, src.transform)
        prof = src.profile | dict(height=int(w.height), width=int(w.width), transform=src.window_transform(w))
        with rasterio.open(dst_path, 'w', **prof) as dst: dst.write(10 ** src.read(1, window=w), 1)

to_utm = Transformer.from_crs('EPSG:4326', 'EPSG:32654', always_xy=True)
centres = {'near plant': (493911, 4141589), 'nw plume': to_utm.transform(140.73, 37.68), 'far sw': to_utm.transform(140.45, 37.30)}
os.makedirs('../_data/windows', exist_ok=True)
wins = {}
for name, (cx, cy) in centres.items():
    wins[name] = f"../_data/windows/{name.replace(' ', '_')}.tif"
    cut_window('../_data/airdose_all_2011.tif', wins[name], cx, cy)

Two permissible levels per window: the 75th and 90th percentiles of its values near the plant, and the 50th and 75th in the other two, where the 90th leaves few units near the boundary. Units near the boundary have between 2% and 30% of their cells above the level: 6 and 9 near the plant (levels 1.17 and 1.48 in log10 of µSv/h), 23 and 16 in the plume (0.69 and 0.74), 10 and 13 far to the south-west (-0.67 and -0.58). The runs use the same levels in µSv/h. The same five strategies as above, 20 seeds:

In [ ]:
levels = {'near plant': (1.17, 1.48), 'nw plume': (0.69, 0.74), 'far sw': (-0.67, -0.58)}
cmp_fk = {}
for name, path in wins.items():
    g = gridder(path, nrows=10, ncols=10)
    for l in levels[name]:
        lvl = 10 ** l
        cmp_fk[f'{name} at {lvl:.3g}'] = compare(Campaign(path, g, score=partial(decisions, level=lvl)), decision_strategies(path, lvl),
                                                 budgets=[150] * 6, seeds=range(20), summary=summarise_decisions)
cmp_fk = pd.concat(cmp_fk, names=['truth'])

Wrong decisions after round 6, averaged over seeds:

In [ ]:
cmp_fk.xs(6, level='round').groupby(['truth', 'strategy']).wrong.mean().unstack().round(1)

`bayes` makes fewer wrong decisions than equal allocation in all six settings. The gap is clear far to the south-west (1.4 against 3.3 at 0.214, 2.2 against 3.2 at 0.263) and within about two standard errors elsewhere; standard errors of the means range from 0.1 to 0.6. It beats the how-to strategy in five settings; near the plant at 30.2 the how-to strategy does better (1.8 against 2.4). The oracle is best everywhere, with the largest gaps near the plant and in the plume at 5.5 (2.8 against 4.8). Neyman allocation is the worst or close to it. No significant negative Moran's I stopped `LINEAR1` in these runs.